## News: daily FinBERT sentiment

For every trading day in 'data/prices.csv', computes the average sentiment of that day's NYT's business articles.
- **news_score**: mean FinBERT score (p_positive - p_negative, between -1 and 1) of the day's articles; 0 if there are none
- **news_count**: number of articles assigned to the day

An article only counts for the day if it was published before 4 PM ET. Articles after count for the next trading day.

Results are saved to `data/news.csv`.

From NYT Archive API (Business news desk) and the pretrained FinBERT model.

** Run 'prices.ipynb' first. The API key is read from '.env' (variable 'NYT_KEY')

In [27]:
# import dependencies
import os, time
from pathlib import Path
import requests
import pandas as pd
import torch
from dotenv import load_dotenv
from transformers import pipeline

In [28]:
load_dotenv()
NYT_KEY = os.getenv("NYT_KEY")       # .env 
print("Key loaded:", NYT_KEY is not None) 

ET = "America/New_York"
trading_days = pd.read_csv("data/prices.csv", index_col=0, parse_dates=True).index

Key loaded: True


## Download Business articles
API returns every article for one month, only keeps the Business desk. 

**Known gap:** the archive for January 2026 stops at 7 Jan (694 articles instead of about 4,000), so 16 trading days have no news.

In [29]:
def get_month(year, month):
    # all NYT articles for a given month, as a list of dicts
    url = f"https://api.nytimes.com/svc/archive/v1/{year}/{month}.json"
    r = requests.get(url, params={"api-key": NYT_KEY}, timeout=120)
    if r.status_code != 200:
        raise RuntimeError(f"NYT request failed for {year}-{month}: status {r.status_code}")
    return r.json()["response"]["docs"]

Path("data/nyt_months").mkdir(parents=True, exist_ok=True)
months = pd.period_range("2021-12", "2026-09", freq="M")

for p in months:
    path = f"data/nyt_months/{p}.csv"
    if os.path.exists(path):  # already downloaded
        continue
   
    docs = get_month(p.year, p.month)
    rows = [{"pub_date": d["pub_date"], 
            "headline": d["headline"]["main"], 
            "abstract": d.get("abstract") or ""} # if they have no abtract
            for d in docs if d.get("news_desk") == "Business"]  

    pd.DataFrame(rows).to_csv(path, index=False)
    print(p, len(rows), "Business articles")
    time.sleep(12) # NYT rate limit

## Each article assigned to a trading day
Dates are parsed as ISO 8601 since it's different in older months "+0000" and from 2026 ".336Z"

In [ ]:
articles = pd.concat([pd.read_csv(f"data/nyt_months/{p}.csv") for p in months], ignore_index=True)
articles["abstract"] = articles["abstract"].fillna("")

articles["time_et"] = pd.to_datetime(articles["pub_date"], utc=True, format="ISO8601").dt.tz_convert(ET)
# keep December 2021 only after the dec 31 close; earlier articles belong before the data starts
articles = articles[articles["time_et"] >= pd.Timestamp("2021-12-31 16:00", tz=ET)].copy()

date = articles["time_et"].dt.tz_localize(None).dt.normalize()
date = date + pd.to_timedelta((articles["time_et"].dt.hour >= 16).astype(int), unit="D")
pos = trading_days.searchsorted(date)         # first trading day on or after each date

keep = pos < len(trading_days)                # drop articles after the last trading day
articles = articles[keep].copy()
articles["trade_date"] = trading_days[pos[keep]]
print(len(articles), "articles assigned to trading days")

16330 articles assigned to trading days


## FinBERT sentiment on articles
Each articles headline and abstract are scored with FinBERT, to give the probability of it being pos, neg, or neutral. Labels are sorted by probability, looked up by name.

In [ ]:
device = "mps" if torch.backends.mps.is_available() else -1    # Apple GPU if available, else CPU
finbert = pipeline("text-classification", model="ProsusAI/finbert", top_k=None, device=device)

def sentiment_score(result):
    """FinBERT output for one text -> p_positive - p_negative."""
    probs = {r["label"]: r["score"] for r in result}
    return probs["positive"] - probs["negative"]

# test
print(sentiment_score(finbert("Stocks surged after strong earnings.")[0]))   
print(sentiment_score(finbert("The bank reported a $2.4 billion loss.")[0]))   

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

0.7037207409739494
-0.8601533779874444


In [33]:
texts = (articles["headline"] + ". " + articles["abstract"]).tolist()
results = finbert(texts, batch_size=32, truncation=True)
articles["score"] = [sentiment_score(r) for r in results]
print(articles["score"].describe().round(3))

count    16330.000
mean        -0.290
std          0.586
min         -0.970
25%         -0.881
50%         -0.331
75%          0.087
max          0.941
Name: score, dtype: float64


## Daily averages
Row for each trading day. If no articles then score = 0 and count = 0.

In [34]:
news = articles.groupby("trade_date").agg(news_score=("score", "mean"), news_count=("score", "size"))
news = news.reindex(trading_days)
news["news_score"] = news["news_score"].fillna(0.0)
news["news_count"] = news["news_count"].fillna(0).astype(int)
news.index.name = "date"

## Checks

In [35]:
print(news.describe().round(3))
print(news["news_count"].sort_values(ascending=False).head(3))     # busiest days
empty = news.index[news["news_count"] == 0]
print(len(empty), "days with no articles:", empty.min().date(), "to", empty.max().date())

       news_score  news_count
count    1190.000    1190.000
mean       -0.287      13.723
std         0.185       5.091
min        -0.857       0.000
25%        -0.413      10.000
50%        -0.290      13.000
75%        -0.171      17.000
max         0.549      43.000
date
2025-04-03    43
2025-04-07    37
2025-01-10    35
Name: news_count, dtype: int64
16 days with no articles: 2026-01-08 to 2026-01-30


## Save
Results are saved to `news.csv` in data folder.

In [36]:
news.to_csv("data/news.csv")
print("Saved", len(news), "rows to data/news.csv")                 # 1190

Saved 1190 rows to data/news.csv
